<a href="https://colab.research.google.com/github/SusanaDataLab/segmentacion-clientes-rfm/blob/main/Online_Retail.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np


In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
data_path = "/content/drive/MyDrive/substack_projects/archive/online_retail_II.csv"
df_online_retail_II = pd.read_csv(data_path)

print("RESUMEN DE DIMENSIONES:")
print("-" * 40)
print(f'- Número de filas: {df_online_retail_II.shape[0]}')
print(f'- Número de columnas: {df_online_retail_II.shape[1]}')


RESUMEN DE DIMENSIONES:
----------------------------------------
- Número de filas: 1067371
- Número de columnas: 8


In [4]:
df_online_retail_II.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [5]:
def informe_calidad(df_online_retail_II):
  resumen = pd.DataFrame({
      'Tipo de dato': df_online_retail_II.dtypes,
      'Nulos': df_online_retail_II.isnull().sum(),
      '% Nulos': (df_online_retail_II.isnull().sum() / len(df_online_retail_II) *100).round(2),
      'Valores únicos': df_online_retail_II.nunique()
  })
  return resumen
print(f"\n=====================")
print(f"INFORME DE CALIDAD")
print(f"=====================")
informe_calidad(df_online_retail_II)


INFORME DE CALIDAD


,Tipo de dato,Nulos,% Nulos,Valores únicos
Invoice,object,0,0.00,53628
StockCode,object,0,0.00,5305
Description,object,4382,0.41,5698
Quantity,int64,0,0.00,1057
InvoiceDate,object,0,0.00,47635
Price,float64,0,0.00,2807
Customer ID,float64,243007,22.77,5942
Country,object,0,0.00,43


In [6]:
# Ver compras con cantidades negativas (devoluciones)
devoluciones = df_online_retail_II[df_online_retail_II['Quantity'] <= 0]
print(f"Total de registros de devoluciones/cancelaciones: {len(devoluciones)}")

# Ver registros con precio cero o negativo
precios_cero = df_online_retail_II[df_online_retail_II['Price'] <= 0]
print(f"Total de registros con precio 0 o menor: {len(precios_cero)}")

Total de registros de devoluciones/cancelaciones: 22950
Total de registros con precio 0 o menor: 6207


### RFM (Recencia, Frecuencia y Valor Monetario)

In [7]:
# 1. Crear copia de trabajo filtrando los nulos de Customer ID
df_clean = df_online_retail_II.dropna(subset=['Customer ID']).copy()

# 2. Filtrar devoluciones/cancelaciones (compras con precios o cantidades <= 0)
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['Price'] > 0)]

# 3. Convertir InvoiceDate a datetime y Customer ID a entero
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
df_clean['Customer ID'] = df_clean['Customer ID'].astype(int)

# 4. Crear columna de Total Venta (Monetary base)
df_clean['Total Sale'] = df_clean['Quantity'] * df_clean['Price']

print(f"Registros originales: {len(df_online_retail_II)}")
print(f"Registros limpios para RFM: {len(df_clean)}")


Registros originales: 1067371
Registros limpios para RFM: 805549


### Generar  tabla **RFM** por cliente

In [8]:
import datetime as dt

# 1. Definir la fecha de referencia para la Recencia
# Sumamos 1 día a la última fecha del dataset para simular que el análisis se hace "hoy"
fecha_referencia = df_clean['InvoiceDate'].max() + dt.timedelta(days=1)

# 2. Agrupar por cliente y calcular los 3 pilares del RFM
rfm = df_clean.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (fecha_referencia - x.max()).days, # Recency: Días sin comprar
    'Invoice': 'nunique',                                       # Frequency: Total de facturas únicas
    'Total Sale': 'sum'                                         # Monetary: Gasto total acumulado
}).reset_index()

# 3. Renombrar columnas
rfm.columns = ['Customer ID', 'Recency', 'Frequency', 'Monetary']

# 4. Asegurar que Customer ID sea un texto sin decimales
rfm['Customer ID'] = rfm['Customer ID'].astype(str)

print("======================================")
print(f"TOTAL DE CLIENTES ÚNICOS IDENTIFICADOS: {len(rfm)}")
print("======================================")
rfm.head()

TOTAL DE CLIENTES ÚNICOS IDENTIFICADOS: 5878


,Customer ID,Recency,Frequency,Monetary
0,12346,326,12,77556.46
1,12347,2,8,5633.32
2,12348,75,5,2019.40
3,12349,19,4,4428.69
4,12350,310,1,334.40


### Crear las puntuaciones RFM (Scoring 1 al 5)

In [11]:
# 1. Asignar puntuaciones del 1 al 5
# Para Recencia: A MENOR días, MEJOR puntuación (por eso labels=[5, 4, 3, 2, 1])
rfm['R_Score'] = pd.qcut(rfm['Recency'], q=5, labels=[5, 4, 3, 2, 1])

# Para Frecuencia y Monetario: A MAYOR valor, MEJOR puntuación (labels=[1, 2, 3, 4, 5])
# Usamos rank(method='first') en Frecuencia para evitar errores con valores repetidos
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5])
rfm['M_Score'] = pd.qcut(rfm['Monetary'], q=5, labels=[1, 2, 3, 4, 5])

# 2. Asignar el Segmento de Negocio según el comportamiento
def segmentar_cliente(df):
    r = int(df['R_Score'])
    f = int(df['F_Score'])

    if r >= 4 and f >= 4:
        return 'Campeones'
    elif r >= 3 and f >= 3:
        return 'Clientes Leales'
    elif r >= 4 and f <= 2:
        return 'Prometedores / Recientes'
    elif r <= 2 and f >= 3:
        return 'En Riesgo / Atención Urgente'
    else:
        return 'Dormidos / Necesitan Reactivación'

rfm['Segmento'] = rfm.apply(segmentar_cliente, axis=1)


print("==============================================")
print("¡SEGMENTACIÓN COMPLETADA!")
print("==============================================")
rfm['Segmento'].value_counts()

¡SEGMENTACIÓN COMPLETADA!


,count
Segmento,
Dormidos / Necesitan Reactivación,1908
Campeones,1482
Clientes Leales,1221
En Riesgo / Atención Urgente,824
Prometedores / Recientes,443


In [ ]:
# from google.colab import files

# # Guardar y descargar la tabla de segmentación RFM por cliente
# rfm.to_csv('rfm_segmentado_final.csv', index=False)
# files.download('rfm_segmentado_final.csv')
